In [1]:
from pathlib import Path

from imagematerials.__main__ import export_summary_netcdf, simulate_stocks
from imagematerials.vehicles.preprocessing import (
    preprocessing,
)
from imagematerials.util import import_from_netcdf, export_to_netcdf
from imagematerials.model import GenericMainModel, GenericMaterials, GenericStocks
from imagematerials.factory import ModelFactory
from imagematerials.maintenance import Maintenance
import prism


In [2]:
base_dir = "../data/raw"
prep_fp = Path("prep_vema2.nc")

In [3]:
if not prep_fp.is_file():
    _, orig_prep_data = preprocessing(base_dir)
    export_to_netcdf(orig_prep_data, prep_fp)
prep_data = import_from_netcdf(prep_fp)
prep_data["weights"] = prep_data.pop("vehicle_weights")
prep_data["battery_materials"]

<xarray.DataArray 'battery_materials' (Cohort: 254, material: 14, battery: 9)> Size: 256kB
array([[[0.05      , 0.0374    ,        nan, ..., 0.24371619,
         0.0374    , 0.00255   ],
        [0.        , 0.        ,        nan, ..., 0.03130767,
         0.07644953, 0.00255   ],
        [0.05      , 0.1003    ,        nan, ..., 0.1247216 ,
         0.1003    , 0.01769956],
        ...,
        [0.1408805 , 0.2147    ,        nan, ..., 0.0074133 ,
         0.2147    , 0.1       ],
        [0.        , 0.        ,        nan, ..., 0.        ,
         0.        , 0.        ],
        [0.        , 0.        ,        nan, ..., 0.        ,
         0.        , 0.        ]],

       [[0.05      , 0.0374    ,        nan, ..., 0.24371619,
         0.0374    , 0.00255   ],
        [0.        , 0.        ,        nan, ..., 0.03130767,
         0.07644953, 0.00255   ],
        [0.05      , 0.1003    ,        nan, ..., 0.1247216 ,
         0.1003    , 0.01769956],
...
        [0.1408805 , 0.2147    ,        nan, ..., 0.0074133 ,
         0.2147    , 0.1       ],
        [0.        , 0.        ,        nan, ..., 0.        ,
         0.        , 0.        ],
        [0.        , 0.        ,        nan, ..., 0.        ,
         0.        , 0.        ]],

       [[0.05      , 0.0374    ,        nan, ..., 0.24371619,
         0.0374    , 0.00255   ],
        [0.        , 0.        ,        nan, ..., 0.03130767,
         0.07644953, 0.00255   ],
        [0.05      , 0.1003    ,        nan, ..., 0.1247216 ,
         0.1003    , 0.01769956],
        ...,
        [0.1408805 , 0.2147    ,        nan, ..., 0.0074133 ,
         0.2147    , 0.1       ],
        [0.        , 0.        ,        nan, ..., 0.        ,
         0.        , 0.        ],
        [0.        , 0.        ,        nan, ..., 0.        ,
         0.        , 0.        ]]])
Coordinates:
  * Cohort    (Cohort) int64 2kB 1807 1808 1809 1810 ... 2057 2058 2059 2060
  * material  (material) <U9 504B 'Aluminium' 'Co' 'Cu' ... 'Steel' 'Ti' 'Wood'
  * battery   (battery) <U16 576B 'LFP' 'LMO' 'LMO/LCO' ... 'NCA' 'NMC' 'NiMH'

In [4]:
# Define the complete timeline, including historic tail
# time_start = prep_data["stocks"].coords["Time"].min().values
time_start = 1960
complete_timeline = prism.Timeline(time_start, 2060, 1)
simulation_timeline = prism.Timeline(1970, 2060, 1)

In [5]:
# Define the coordinates of all dimensions.
Region = list(prep_data["stocks"].coords["Region"].values)
Time = [t for t in complete_timeline]
Cohort = Time
Type = list(prep_data["stocks"].coords["Type"].values)
material = list(prep_data["material_fractions"].coords["material"].values)
battery = list(prep_data["battery_materials"].coords["battery"].values)

# Create
main_model_normal = GenericMainModel(
    complete_timeline, Region=Region, Time=Time, Cohort=Cohort, Type=Type, prep_data=prep_data,
    compute_materials=True, compute_battery_materials=True, compute_maintenance_materials=False, 
    material=material, battery = battery)

In [6]:
main_model_normal.simulate(simulation_timeline)

In [16]:
main_model_normal.battery_model.inflow_battery[2020]

Magnitude,[[[nan nan nan ... nan nan nan] [nan nan nan ... nan nan nan] [nan nan nan ... nan nan nan] ... [nan nan nan ... nan nan nan] [nan nan nan ... nan nan nan] [nan nan nan ... nan nan nan]] [[nan nan nan ... nan nan nan] [nan nan nan ... nan nan nan] [nan nan nan ... nan nan nan] ... [nan nan nan ... nan nan nan] [nan nan nan ... nan nan nan] [nan nan nan ... nan nan nan]] [[nan nan nan ... nan nan nan] [nan nan nan ... nan nan nan] [nan nan nan ... nan nan nan] ... [nan nan nan ... nan nan nan] [nan nan nan ... nan nan nan] [nan nan nan ... nan nan nan]] ... [[nan nan nan ... nan nan nan] [nan nan nan ... nan nan nan] [nan nan nan ... nan nan nan] ... [nan nan nan ... nan nan nan] [nan nan nan ... nan nan nan] [nan nan nan ... nan nan nan]] [[nan nan nan ... nan nan nan] [nan nan nan ... nan nan nan] [nan nan nan ... nan nan nan] ... [nan nan nan ... nan nan nan] [nan nan nan ... nan nan nan] [nan nan nan ... nan nan nan]] [[nan nan nan ... nan nan nan] [nan nan nan ... nan nan nan] [nan nan nan ... nan nan nan] ... [nan nan nan ... nan nan nan] [nan nan nan ... nan nan nan] [nan nan nan ... nan nan nan]]]
Units,count


In [8]:
main_model_factory = ModelFactory(
    prep_data, complete_timeline
    ).add(GenericStocks
    ).add(GenericMaterials
    ).finish()

In [9]:
main_model_factory.simulate(simulation_timeline)